In [ ]:
%load_ext autoreload
%autoreload 2

import glob, os, sys

import numpy as np
import matplotlib.pyplot as plt

import torch
torch.serialization.add_safe_globals
from torch.utils.data import TensorDataset, DataLoader
#from tabulate import tabulate
from pytorch_networks_convae import *
import argparse
from datasetio import *
import copy
from scaler import *
import time
import pickle 

In [ ]:
# define here
data_dir = ""
nn_dir = ""
# or
# import from a file
from paths import *

In [ ]:
colors = ["r-", "g-", "b-", "c-", "y-", "m-","k-",
          "r--", "g--", "b--", "c--", "y--", "m--","k--",
          "r.", "g.", "b.", "c.", "y.", "m.",
          "r:", "g:", "b:", "c:", "y:", "m:"] 

NUM_COLORS = 36
cm = plt.get_cmap('gist_rainbow')

para_counts = []
min_mae_u = []
min_mae_v = []
min_mae_p = []
labels = []

debug = False
# kernel, symmetry, layers, loss_scale loss_derivative, filters, "loss", a_bound, blurr, batch size, l2_reg, padding, factor, levels
combs = [        
    
            [[5],   [False],      [6], [[True,True]],       [16],      ["mass", "curl"], [10],   [False], [16], [0.0], 
             ["mass126_newfluidnet"], ["learned"], [2], [5]],
    

            [[5],   [False],      [6],     [[True,True]],       [16],      ["curl"], [10],   [False], [16], [0.0], ["fluidnet", "newfluidnet"], 
             ["learned"], [2], [5]],
    
            [[5],   [False],      [6],     [[True,True]],       [16],      ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
             ["learned","replicate","zeros"], [2], [5]],
            
            #[[5],   [True,False], [6],    [[True,True]],       [16],      ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
            # ["learned"], [2], [5]],
            
            [[3,5], [False],      [6],     [[True,True]],       [16],      ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
             ["learned"], [2], [5]],
            
            [[5],   [False],      [6],    [ [True,True]],       [8,16,32], ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
             ["learned"], [2], [5]],

            [[5],   [False],      [6],     [[True,True]],       [16,64], ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
             ["zeros"], [2], [5]],
            
            [[5],   [False],      [6],    [[True,True], [True,False], [False,True], [False,False]], 
             [16],      ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
             ["learned"], [2], [5]],
            
            [[5],   [False],      [4,6], [[True,True]],       [16],      ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
            ["learned"], [2], [5]],
            
            [[5],   [False],      [6], [[True,True]],       [16],      ["curl", "mass", "mae"], [10],   [False], [16], [0.0], 
             ["newfluidnet"], ["learned"], [2], [5]],
            
            #[[5],   [False],      [6], [[True,True]],       [16],      ["curl"], [10],   [False], [16], [0.0], ["newfluidnet"], 
            # ["learned"], [2,3], [5]],
        ]


for comb in combs:
    fig = plt.figure(figsize=(12,5),dpi=320)
    ax = {}
    cntr = 0
    for k in range(8):
        ax[cntr] = fig.add_subplot(2,4,cntr+1)
        ax[cntr].set_prop_cycle('color', [cm(1.*i/NUM_COLORS) for i in range(NUM_COLORS)])
        cntr += 1

    
    counter = 0
    act_fn = "gelu"
    dilation = 1
    use_skip = False
    blurr = False
    d_r = 0.0
    p_pred = False
    advect = False

    for ker in comb[0]: 
        for sym in comb[1]:
            for rep in comb[2]:
                for loss_scale, loss_derivative in comb[3]:
                    for fil in comb[4]:
                        for loss_type in comb[5]:
                            for a_bound in comb[6]:
                                for blurr in comb[7]:
                                    for batch_size in comb[8]:
                                        for l2 in comb[9]:
                                            for network in comb[10]:
                                                for r_p in comb[11]:
                                                    for factor in comb[12]:
                                                        for levels in comb[13]:
                                                            
                                                            bbatch_size = batch_size
                                                            lev = levels
                                                            reps = rep
                                                            if factor==3:
                                                                lev = 3                                                                
                                                            
                                                            if network=="unet":
                                                                roll = 1
                                                                reps = 3
                                                            else:
                                                                if network=="multiscalenewfluidnet":
                                                                    filt = 4
                                                                    reps = 1
                                                                else:
                                                                    filt = fil
                                                                if fil==64:
                                                                    bbatch_size = 8
                                                                    reps = 4                                                        

                                                            r_pp = r_p
                                                            symm = sym
                                                            f_nn =  network + "_levels_" + str(lev) + "_" + act_fn + \
                                                                    "_" + str(filt) + "_" + r_pp + "_" + loss_type + \
                                                                    "_" + str(symm) + "_ab" + str(a_bound) + "_b" + str(bbatch_size) + \
                                                                    "_r" + str(reps) + "_k" + str(ker) + "_fa" + str(factor) + \
                                                                    "_ad" + str(advect) + "_p_pred" + str(p_pred) +\
                                                                    "_l2" + str(l2) + "_l_sc" + str(loss_scale) +\
                                                                    "_l_de" + str(loss_derivative) + "_deb" + str(debug)
                                                            if network == "unet":
                                                                f_nn += "_roll" + str(1)
                                                                
                
                                                            if blurr:
                                                                f_nn += "_blurr"
                                                            
                                                            layers = []
                                                            loss_layer = []
                                                            loss_cv_layer = []
                                                            _nn_dir = nn_dir + f_nn + "/"
                                                                        
                                                            with open(_nn_dir + "fluidnet_uvpT.txt") as fw:
                                                                lines = fw.readlines()
                                                            fw.close()
                                                            loss_u       = []
                                                            loss_v       = []
                                                            loss_p       = []
                                                            loss_mass    = []
                                                            loss_cv_u    = []
                                                            loss_cv_v    = []
                                                            loss_cv_p    = []
                                                            loss_cv_mass = []
                                                            
                                                            for l in lines[1:]:
                                                                ll    = l[l.index("[")+1:l.index("],[")].split(",")
                                                                l_r   = l[l.index("],[")+3:]
                                                                ll_cv = l_r[:l_r.index("],")].split(",")                   
                                                        
                                                                loss_u.append([float(ll[0])])
                                                                loss_v.append([float(ll[1])])
                                                                loss_p.append([float(ll[3])])    
                                                                loss_mass.append([float(ll[4])+1e-16])
                                                                
                                                                loss_cv_u.append([float(ll_cv[0])])
                                                                loss_cv_v.append([float(ll_cv[1])])
                                                                loss_cv_p.append([float(ll_cv[3])])
                                                                loss_cv_mass.append([float(ll_cv[4])+1e-16])
                                                        
                                                            #layers.append(int(f_nn[-1]))
                                                            #loss_layer.append([min(loss_u), min(loss_v), min(loss_p)])
                                                            #loss_cv_layer.append([min(loss_cv_u), min(loss_cv_v), min(loss_cv_p)])
                                        
                                                            label = network[::5] + "f" + str(fil) + "_s" + str(symm) + "_r" + str(rep) \
                                                                    + "_k" + str(ker) + "_bl" + str(blurr) + "_lt" \
                                                                    + loss_type + "_l_sc" + str(loss_scale) + "_l_de" + str(loss_derivative) \
                                                                    + "_ab" + str(a_bound) + "_p" + r_pp \
                                                                    + "_l" + str(lev) + "_fac" + str(factor)
                                                            
                                                            iterations = np.arange(1,len(loss_u)+1)
                                                            
                                                            ax[0].plot(iterations, loss_u, colors[counter], label=label)
                                                            ax[1].plot(iterations, loss_v, colors[counter], label=label)
                                                            ax[2].plot(iterations, loss_p, colors[counter], label=label)
                                                            ax[3].plot(iterations, loss_mass, colors[counter], label=label)
                                                            
                                                            ax[4].plot(iterations, loss_cv_u, colors[counter], label=label)
                                                            ax[5].plot(iterations, loss_cv_v, colors[counter], label=label)
                                                            ax[6].plot(iterations, loss_cv_p, colors[counter], label=label)
                                                            ax[7].plot(iterations, loss_cv_mass, colors[counter], label=label)
                                        
                                                            counter += 1  
                                        
                                                            #para_counts.append(get_model_parameters(fil, rep, ker, sym))
                                                            #min_mae_u.append(min(loss_cv_u))
                                                            #min_mae_v.append(min(loss_cv_v))
                                                            #min_mae_p.append(min(loss_cv_p))
                                                            #labels.append(label)

                                                            #print(f_nn, np.min(np.asarray(loss_cv_u)), np.min(np.asarray(loss_cv_v)))


    for var_ind, var in enumerate(["u", "v", "T", "mass"]):
        ax[var_ind].set_title(var + " Train")
        ax[var_ind+4].set_title(var + " CV")
        ax[var_ind].set_ylabel("Mean Absolute Error")
        ax[var_ind+4].set_ylabel("Mean Absolute Error")
        ax[var_ind].set_xlabel("Epochs")
        ax[var_ind+4].set_xlabel("Epochs")
                                        
    for ax_ind in range(8):
        ax[ax_ind].set_yscale("log")
        
        if ax_ind in [0,1,4,5] and "mass126" not in f_nn:
            if debug:
                ax[ax_ind].set_ylim([2e-4,1e-1])
            else:
                ax[ax_ind].set_ylim([2e-4,1e-2])
        #ax[ax_ind].set_xscale("log")
        #if ax_ind != 3 and ax_ind != 7:
            #ax[ax_ind].set_ylim([10,500])
            #ax[ax_ind].set_xlim([0, 20])
    ax[2].legend(prop={'size': 6})
    #ax[6].legend(prop={'size': 4})
    plt.tight_layout()
    plt.show()

In [ ]:
gpu_number = 1
device = torch.device("cuda:" + str(gpu_number)) if torch.cuda.is_available() else torch.device("cpu")
#device = torch.device("cpu")

act_fn = "gelu"

network = "newfluidnet"
repeats = 6
levels = 5
kernel = 5
c_h    = 16
epoch  = 81
blurr = False
debug = False
loss_scale = True
loss_derivative = True
batch_size = 16
loss_type = "curl"
a_bound   = 10
r_p = "learned"
use_symm  = False 
factor = 2

l2  = 0.0
d_r = 0.0

dilation = 1
use_skip = False
scale = True
p_pred = False
noise = 0.0

advect = False
spectral_conv = False

f_nn   =    network + "_levels_" + str(levels) + "_" + act_fn + \
            "_" + str(c_h) + "_" + r_p + "_" + loss_type +  \
            "_" + str(use_symm) + "_ab" + str(a_bound) + "_b" + str(batch_size) + \
            "_r" + str(repeats) + "_k" + str(kernel) + "_fa" + str(factor) + \
            "_ad" + str(advect) + "_p_pred" + str(p_pred) + \
            "_l2" + str(l2) + "_l_sc" + str(loss_scale) + "_l_de" + str(loss_derivative) + "_deb" + str(debug)  

if blurr:
    f_nn += "_blurr"
    
nn_dir = nn_dir + f_nn + "/"

if network=="fluidnet" or network=="newfluidnet":
    c_i = 7
    c_o = 3
elif network=="newfluidnet":
    c_i = 7
    c_o = 3
elif network == "ifluidnet":
    c_i = 9
    c_o = 3
elif network == "convae":
    c_i = 3
    c_o = 3
elif network == "unet":
    c_i = 11
    c_o = 4
    if not p_pred:
        c_i -= 1

if loss_type == "curl":
    c_o -= 1
if not p_pred:
    c_o -= 1

if network=="fluidnet" or network == "ifluidnet":
    model_uvp = FluidNet(levels, c_i, c_h, c_o, device, act_fn, r_p, loss_type, 
                         use_symm=use_symm, dilation=dilation, a_bound=a_bound,
                         repeats=repeats, use_skip=use_skip, f=kernel, p_pred=p_pred, blurr=blurr).double().to(device)

    ts = 1
    ts_net = TS(model_uvp, ad=None, device=device, ts=ts, advection_scheme=0, 
                scale=scale, p_pred=p_pred, net=network).double().to(device)

elif network=="newfluidnet":
    model_uvp = NewFluidNet(levels, c_i, c_h, c_o, device, act_fn, r_p, loss_type, 
                         use_symm=use_symm, dilation=dilation, a_bound=a_bound,
                         repeats=repeats, use_skip=use_skip, f=kernel, p_pred=p_pred, 
                            blurr=blurr, factor=factor).double().to(device)
    
    ts = 1
    ts_net = TS(model_uvp, ad=None, device=device, ts=ts, advection_scheme=0, 
                scale=scale, p_pred=p_pred, net=network).double().to(device)

        
elif network == "unet":
    model_uvp = Unet(levels, c_i, c_h, c_o, device, act_fn, r_p, loss_type, 
                         use_symm=use_symm, dilation=dilation, a_bound=a_bound,
                         repeats=repeats, use_skip=use_skip, f=kernel, p_pred=p_pred).double()

print(count_parameters(model_uvp))

if debug:
    model_uvp.load_state_dict(torch.load(nn_dir + "fluidnet_uvp.pt", map_location=device))
else:
    model_uvp.load_state_dict(torch.load(nn_dir + str(epoch) + "_fluidnet_uvp.pt", map_location=device))

torch.compile(ts_net)
ts_net.eval()

In [ ]:
load_limited_data = True
save_preds = []

sims = torch.load(data_dir + "/sims.pt", weights_only=False)

a_min = []
a_max = []
u_min = []
u_max = []
v_min = []
v_max = []

for an in ["test"]:
    x_list = []
    y_list = []
    total_samples = 0
    
    for si, sim in enumerate(sims):
        ignr, ignr, raq, fkt, fkp, gr, ar, ignr = sim
        if debug:
            check = si==0
        else:
            check = (sim[1] == an and (si==116 or si == 127)) # raq in [8.75081696]) #0.526931, 6.79733173, 3.66563052])
        if check:
            print(tabulate([["num", "dataset", "raq", "fkt", "fkp", "gr", "ar"],
                            sim[:-1]
                           ]))

            py_dir = data_dir + "/" + sim[1] + "/sim_" + str(sim[0])
                
            raq_nd = torch.tensor((raq-0.12624371)/(9.70723344-0.12624371), dtype=torch.float64)
            fkt_nd = torch.tensor((np.log10(fkt)-6.00352841978384)/(
                9.888820429862925-6.00352841978384), dtype=torch.float64)
            fkp_nd = torch.tensor((np.log10(fkp)-0.005251646002323797)/(
                1.9927988938926755-0.005251646002323797), dtype=torch.float64)

            fkt = torch.tensor(fkt, dtype=torch.float64)
            fkp = torch.tensor(fkp, dtype=torch.float64)
            
            xcc    = torch.load(py_dir + "/xc.pt", weights_only=False)
            ycc    = torch.load(py_dir + "/yc.pt", weights_only=False)
            xcc    = xcc.view(1,1,xcc.shape[0],xcc.shape[1])
            ycc    = ycc.view(1,1,ycc.shape[0],ycc.shape[1])

            xcc[:,:,:,0]  = 0.0
            xcc[:,:,:,-1] = 4.0
            ycc[:,:,0,:]  = 0.0
            ycc[:,:,-1,:] = 1.0

            sdf = torch.zeros_like(ycc)
            sdf[:,:,0,:]  = 1.
            sdf[:,:,-1,:] = 1.
            sdf[:,:,:,0]  = 1.
            sdf[:,:,:,-1] = 1.

            sdf2 = torch.ones_like(ycc)
            sdf2[:,:,0,:]  = 0.
            sdf2[:,:,-1,:] = 0.
            sdf2[:,:,:,0]  = 0.
            sdf2[:,:,:,-1] = 0.

            take_every = 1

            if load_limited_data:
                u  = torch.load(py_dir + "/e" + str(take_every) + "_uprev_data_select_snaps.pt", weights_only=False)[1:,...]
                v  = torch.load(py_dir + "/e" + str(take_every) + "_vprev_data_select_snaps.pt", weights_only=False)[1:,...]
                Tprev = torch.load(py_dir + "/e" + str(take_every) + "_Tprev_data_select_snaps.pt", weights_only=False)[1:,...]
                i_vec = np.arange(u.shape[0])
                i_vec = [2, 3, 6]

            else:
                u  = torch.load(py_dir + "/e" + str(take_every) + "_uprev_data.pt", weights_only=False)[1:,...]
                v  = torch.load(py_dir + "/e" + str(take_every) + "_vprev_data.pt", weights_only=False)[1:,...]
                Tprev = torch.load(py_dir + "/e" + str(take_every) + "_Tprev_data.pt", weights_only=False)[1:,...]
                if p_pred:
                    p  = torch.load(py_dir + "/e" + str(take_every) + "_pprev_data.pt", weights_only=False)[1:,...]
                
                i_vec = [1, 2, 3, int(u.shape[0]/10), int(u.shape[0]/50), u.shape[0]-2]

                u_min = torch.amin(torch.abs(u),axis=(1,2,3))
                u_max = torch.amax(torch.abs(u),axis=(1,2,3))
                v_min = torch.amin(torch.abs(v),axis=(1,2,3))
                v_max = torch.amax(torch.abs(v),axis=(1,2,3))
    
                i_vec_cands = [  torch.argmin(v_max),
                                 torch.argmin(torch.abs(v_min)),
                                 torch.argmax(v_max),
                                 torch.argmax(torch.abs(v_min)),
                                 torch.argmin(u_max),
                                 torch.argmin(torch.abs(u_min)),
                                 torch.argmax(u_max),
                                 torch.argmax(torch.abs(u_min)),
                                 
                                ]
    
                i_vec = []
                for i in i_vec_cands:
                    if i not in i_vec:
                        i_vec.append(i)

                if si == 116:
                    i_vec = [0]
                elif si == 127:
                    i_vec = [5182]
            
            for i in i_vec: # 5, 20, int(u.shape[0]/10), int(u.shape[0]/40), u.shape[0]-1]: 
                Tp = Tprev[i:i+1,...]
                
                t0 = time.time()
                _, _, u_pred,v_pred,p_pred,V = ts_net(Tp, sdf, sdf2, ycc, raq_nd, fkt_nd, fkp_nd, raq, fkt, fkp, xcc, ycc)
                
                t1 = time.time()
                print("Inference took: " + str(t1-t0))
                
                if p_pred:
                    y      = torch.cat((u[i:i+1,...],v[i:i+1,...],p[i:i+1,...]), axis=1)
                    y_base = torch.cat((u[i-1:i,...],v[i-1:i,...],p[i-1:i,...]), axis=1)
                    y_pred = torch.cat((u_pred, v_pred, p_pred), axis=1)
                else:
                    y      = torch.cat((u[i:i+1,...],v[i:i+1,...]), axis=1)
                    y_base = torch.cat((u[i-1:i,...],v[i-1:i,...]), axis=1)
                    y_pred = torch.cat((u_pred, v_pred), axis=1)


                save_preds.append([u[i:i+1,...].cpu().detach().numpy(),
                                   v[i:i+1,...].cpu().detach().numpy(),
                                   u_pred.cpu().detach().numpy(),
                                   v_pred.cpu().detach().numpy()])
                
                y[:,0:1,...]      = scale_var(y[:,0:1,...], raq, fkt, fkp, "uprev") 
                y[:,1:2,...]      = scale_var(y[:,1:2,...], raq, fkt, fkp, "vprev") 
                y_base[:,0:1,...] = scale_var(y_base[:,0:1,...], raq, fkt, fkp, "uprev") 
                y_base[:,1:2,...] = scale_var(y_base[:,1:2,...], raq, fkt, fkp, "vprev") 
                y_pred[:,0:1,...] = scale_var(y_pred[:,0:1,...], raq, fkt, fkp, "uprev") 
                y_pred[:,1:2,...] = scale_var(y_pred[:,1:2,...], raq, fkt, fkp, "vprev") 
                
                
                #V = torch.clip(V,1e-8,1)
                fig = plt.figure(figsize=(15,5),dpi=160)
                ax = fig.add_subplot(1,2,1)
                cax = ax.tricontour(xcc.flatten(), ycc.flatten(), (torch.log10(V)/8).cpu().numpy().flatten(),levels=16) 
                fig.colorbar(cax)
                
                ax = fig.add_subplot(1,2,2)
                cax = ax.tricontour(xcc.flatten(), ycc.flatten(), Tp.flatten(),levels=16) 
                fig.colorbar(cax)
                
                plt.tight_layout()
                plt.show()
                
                if 1==1: #i in : #:
                    if p_pred:
                        var = ["u", "v", "p", "mass"]
                    else:
                        var = ["u", "v", "mass"]
                    levels = 16
                    fig = plt.figure(figsize=(24,3*len(var)),dpi=640)
                    ax = {}
                    cntr = 0
                    for var_ind in range(len(var)):
                        for k in range(3):
                            ax[cntr] = fig.add_subplot(len(var),3,cntr+1)
                            cntr += 1

                    figb = plt.figure(figsize=(24,3*(len(var)-1)),dpi=640)
                    axb = {}
                    cntr = 0
                    for var_ind in range(len(var)-1):
                        for k in range(4):
                            axb[cntr] = figb.add_subplot(len(var)-1,4,cntr+1)
                            cntr += 1
    
                    cntr = 0
                    for var_ind in range(len(var)):  
                        if var[var_ind] == "mass":
                            u_s_t = y[:,0:1,...]
                            v_s_t = y[:,1:2,...]
                            u_s_p = y_pred[:,0:1,...]
                            v_s_p = y_pred[:,1:2,...]

                            bc = True #True if network=="newfluidnet" else False
                            z_t   = get_mass(u_s_t.cpu(), v_s_t.cpu(), False).cpu().numpy()
                            z_p   = get_mass(u_s_p.cpu(), v_s_p.cpu(), bc).detach().numpy()
                            z_b   = z_t
                            xc    = copy.copy(xcc[...,1:-1,1:-1])
                            yc    = copy.copy(ycc[...,1:-1,1:-1])
                            levels = 2 #16
                        else:
                            z_b = y_base[:,var_ind,...].cpu().detach().numpy()
                            z_t = y[:,var_ind,...].cpu().detach().numpy()
                            z_p = y_pred[:,var_ind,...].cpu().detach().numpy()
                            xc    = copy.copy(xcc)
                            yc    = copy.copy(ycc)
                            
                            vmin = z_t.min()
                            vmax = z_t.max()
                            levels = np.linspace(vmin,vmax,16)
                            
                        mae_pred = np.mean(np.abs(z_t-z_p))
                        mae_base = np.mean(np.abs(z_t-z_b))
                        print("mae prediction " + str(var[var_ind]) + "    : " + str(mae_pred) 
                              )
                        print("mae baseline " + str(var[var_ind]) + "      : " + str(mae_base))


                        print("baseline/prediction : " + str(mae_base/mae_pred))
                        print()

                        cax = ax[cntr].tricontour(xc.flatten(), yc.flatten(), z_t.flatten(),
                                             levels=levels)
                        ax[cntr].set_title("True " + var[var_ind])
                        cntr += 1
                        fig.colorbar(cax)
                        
                        cax = ax[cntr].tricontour(xc.flatten(), yc.flatten(), z_p.flatten(),
                                             levels=levels) #levels)
                        ax[cntr].set_title("Prediction")
                        cntr += 1
                        fig.colorbar(cax)

                        #diff = np.log10(np.abs(z_t-z_p)/np.abs(z_t+1e-16) + 1e-16)
                        diff = (z_t-z_p) /(np.amax(abs(z_t)))*100
                        cax = ax[cntr].tricontourf(xc.flatten(), yc.flatten(), diff.flatten(),
                                             levels=16) #np.linspace(-0.01,0.01,16))
                        ax[cntr].set_title("Difference/Maximum %")
                        cntr += 1
                        fig.colorbar(cax)

                        if var[var_ind] != "mass":
                            bnd_colors = ["r", "g", "b", "k"]
                            for bnd_ind in [1]:
                                axb[0+4*var_ind].plot(z_t[0,:,bnd_ind], bnd_colors[bnd_ind]+'-')
                                axb[1+4*var_ind].plot(z_t[0,:,-1-bnd_ind], bnd_colors[bnd_ind]+'-')
                                axb[2+4*var_ind].plot(z_t[0,-1-bnd_ind,:], bnd_colors[bnd_ind]+'-')
                                axb[3+4*var_ind].plot(z_t[0,bnd_ind,:], bnd_colors[bnd_ind]+'-')
    
                                axb[0+4*var_ind].plot(z_p[0,:,bnd_ind], bnd_colors[bnd_ind]+'--')
                                axb[1+4*var_ind].plot(z_p[0,:,-1-bnd_ind], bnd_colors[bnd_ind]+'--')
                                axb[2+4*var_ind].plot(z_p[0,-1-bnd_ind,:], bnd_colors[bnd_ind]+'--')
                                axb[3+4*var_ind].plot(z_p[0,bnd_ind,:], bnd_colors[bnd_ind]+'--')

                            fig.legend()
                            
                        #fig.suptitle(var[var_ind])
                    
                    fig.tight_layout()
                    figb.tight_layout()
                    
                    #fig.savefig(nn_dir + str(si) + "_" + str(i) + ".pdf")
                    #figb.savefig(nn_dir + str(si) + "_" + str(i) + "_bc.pdf")
                    fig.show()

                    figb.tight_layout()
                    figb.show()

                    
                    '''
                    mae_model    = torch.mean(torch.abs(y[:,-1:,:,:]-y_pred[:,-1:,:,:])).item()
                    mae_baseline = torch.mean(torch.abs(y[:,-1:,:,:]-Tp)).item()
                    print("prediction: " + str(mae_model))
                    print("baseline  : " + sstr(mae_baseline))
                    '''


In [ ]:
#with open("Paper/FiguresData/vel_preds.pkl", "wb") as file:
#    pickle.dump(save_preds, file)

In [ ]:
## Parameter / inference time calculations

gpu_number = 1
device = torch.device("cuda:" + str(gpu_number)) if torch.cuda.is_available() else torch.device("cpu")

act_fn = "gelu"

levels = 5
kernel = 5
epoch  = 70
blurr = False
debug = False
loss_scale = True
batch_size = 16
loss_type = "curl"
a_bound   = 10
use_symm  = False 
factor = 2

l2  = 0.0
d_r = 0.0

dilation = 1
use_skip = False
scale = True
p_pred = False
noise = 0.0

advect = False
spectral_conv = False

for network, repeats, c_h, r_p in [

                                   #["multiscalenewfluidnet", 1, 4, "learned"],
    
                                   ["newfluidnet", 6, 16, "learned"],
                                   ["newfluidnet", 6, 16, "zeros"],
                                   #["unet", 3, 6, "learned"],  
                                  # ["unet", 3, 16, "learned"] ,

                                   #["newfluidnet", 4, 64, "zeros"],
                                   #["unet", 3, 16, "replicate"],  
                                   #["unet", 3, 64, "replicate"],

                                    
                                   
                                   ]:

    torch.cuda.empty_cache()
    
    if network=="newfluidnet" or network=="multiscalenewfluidnet":
        c_i = 7
        c_o = 1
    elif network == "unet":
        c_i = 11
        c_o = 2

    inp = torch.randn(1, c_i, 128, 506).double().to(device)
        
    if network=="newfluidnet":
        model_uvp = NewFluidNet(levels, c_i, c_h, c_o, device, act_fn, r_p, loss_type, 
                             use_symm=use_symm, dilation=dilation, a_bound=a_bound,
                             repeats=repeats, use_skip=use_skip, f=kernel, p_pred=p_pred, 
                                blurr=blurr, factor=factor).double().to(device)
            
    elif network == "unet":
        model_uvp = Unet(levels, c_i, c_h, c_o, device, act_fn, r_p, loss_type, 
                             use_symm=use_symm, dilation=dilation, a_bound=a_bound,
                             repeats=repeats, use_skip=use_skip, f=kernel, p_pred=p_pred).double().to(device)

    elif network == "multiscalenewfluidnet":
        models = torch.nn.ModuleList()
        for _ in [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e+0, 1e+1]:
            models.append(HalfNewFluidNet(levels, c_i, c_h, c_o, device, act_fn, r_p, loss_type, 
                             use_symm=use_symm, dilation=dilation, a_bound=a_bound,
                             repeats=repeats, use_skip=use_skip, f=kernel, p_pred=p_pred, 
                                blurr=blurr, factor=factor).double().to(device))
        model_uvp = MultiScaleNewFluidNet(nets=models, loss_type=loss_type, 
                                          device=gpu_number, scales=[1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e+0, 1e+1],
                                          p_pred=False).double()

    torch.compile(model_uvp)
    model_uvp.eval()
    times = []
    with torch.no_grad():
        for _ in range(500):# 500):
            t0 = time.time()
            model_uvp(inp)
            t1 = time.time()
            times.append(t1-t0)
        
    print(network, repeats, c_h, r_p, count_parameters(model_uvp), np.mean(np.asarray(times)))